# LCEL Advanced - Runnable 심화

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## RunnableLambda
- 일반 파이썬 함수를 Runnable 객체로 감싸는 도구
- 일반 함수는 단독으로 LCEL 체인 안에 직접 연결하기 어려우므로 사용한다.
- `invoke`, `batch`, `stream`, `|` 조합 등의 Runnable 공통 실행 방식을 사용할 수 있다.

In [2]:
from langchain_core.runnables import RunnableLambda

# 문자열의 길이를 반환하는 일반 파이썬 함수를 Runnable로 감싼다.
runnable = RunnableLambda(lambda text: len(text))

# invoke: 하나의 입력을 받아 하나의 결과를 반환한다. (기본적인 Runnable 실행 방식)
runnable.invoke("안녕하세요")

5

In [3]:
def count_chars(text: str) -> int:
    return len(text)

runnable = RunnableLambda(count_chars)

runnable.invoke("LangChain")

9

In [4]:
# batch() : 여러 입력을 한 번에 처리한다.
# 입력 리스트를 전달하면 각 입력에 대해 Runnable을 실행하고, 결과 리스트를 반환한다.
runnable.batch(['안녕하세요', '잘가', 'LangChain'])

[5, 2, 9]

In [6]:
# stream() : 결과를 한 번에 반환하지 않고 조각 단위로 받을 때 사용한다.
import time

def stream_text(text: str):
    for char in text:
        time.sleep(0.05)
        yield char

runnable = RunnableLambda(stream_text)

for chunk in runnable.stream("LCEL stream은 결과를 조각 단위로 받을 때 사용한다."):
    print(chunk, end="", flush=True)

LCEL stream은 결과를 조각 단위로 받을 때 사용한다.

In [ ]:
# invoke는 결과를 모두 모아서 한 번에 반환한다.
result = runnable.invoke("LCEL stream은 결과를 조각 단위로 받을 때 사용한다.")
print(result)

LCEL stream은 결과를 조각 단위로 받을 때 사용한다.


## Composition - RunnableSequence
- `RunnableSequence`는 여러 Runnable을 순서대로 실행하는 구성이다.
- 단, 실제 코드에서는 `|` 연산자를 사용해 연결하는 경우가 많다.

In [11]:
from langchain_core.runnables import RunnableSequence

# 입력 값을 리스트로 3번 반복한다.
repeat_three_times = RunnableLambda(lambda x: [x] * 3)

# 입력 값을 딕셔너리 형태로 감싼다.
to_dict = RunnableLambda(lambda x: {"result" : x})

# RunnableSequence 객체를 통해 연결
sequence_chain = RunnableSequence(repeat_three_times, to_dict)
sequence_chain.invoke("A")

{'result': ['A', 'A', 'A']}

In [12]:
# 같은 체인을 | 연산자로 더 간결하게 표현할 수 있다.
pipe_chain = repeat_three_times | to_dict
pipe_chain.invoke("B")

{'result': ['B', 'B', 'B']}

## RunnableLambda 를 활용한 라우팅

In [13]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

llm = init_chat_model("openai:gpt-4.1-mini")
output_parser = StrOutputParser()

In [14]:
# 수학 선생님 체인
math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단계적인 풀이를 수식(LaTex)과 함께 작성해주세요.\n\n{question}')
math_chain = math_prompt | llm | output_parser

# 기본 체인
default_prompt = PromptTemplate.from_template('당신은 친절하고, 감성적인 공감 능력이 좋은 챗봇입니다. 다음 질문에 답해주세요.\n\n{question}')
default_chain = default_prompt | llm | output_parser

In [15]:
def route_chain(input_data: dict):
    # 입력 내용을 보고 실행할 체인을 반환한다.
    question = input_data['question']
    math_keywords = ["계산", "수학", "+", "-", "*", "/", "더하기", "빼기", "곱하기", "나누기"]

    if any(keyword in question for keyword in math_keywords):
        return math_chain
    
    return default_chain

router_chain = RunnableLambda(route_chain)

# 수학 질문
print(router_chain.invoke({'question' : '1254 * 3 + 50 이거 좀 계산해줘'}))

# 그외 질문
print(router_chain.invoke({'question' : '나 오늘 부장님한테 깨졌어. 우울하다ㅠㅠ'}))

문제를 단계적으로 풀어보겠습니다.

주어진 식은:
\[
1254 \times 3 + 50
\]

1단계: 먼저 곱셈을 계산합니다.
\[
1254 \times 3 = 3762
\]

2단계: 곱한 결과에 50을 더합니다.
\[
3762 + 50 = 3812
\]

답은:
\[
\boxed{3812}
\]
아이고, 정말 속상했겠어요. 부장님한테 그런 말을 들으면 마음이 많이 무너질 수밖에 없죠. 많이 힘들고 우울한 기분 이해해요. 그래도 당신이 그 상황 속에서도 최선을 다했을 거라는 걸 믿어요. 조금 쉬면서 마음 돌볼 시간 꼭 가지셨으면 좋겠어요. 언제든 이야기하고 싶으면 편하게 말해줘요. 제가 곁에 있을게요.


## Composition - RunnableParallel
- `RunnalbeParallel` 은 하나의 입력을 여러 Runnable에 전달하고, 각 Runnable의 결과를 딕셔너리 형태로 모아 반환한다.

In [16]:
from langchain_core.runnables import RunnableParallel

square = RunnableLambda(lambda x: x ** 2)
cube = RunnableLambda(lambda x: x ** 3)
is_even = RunnableLambda(lambda x: x % 2 == 0)

parallel_chain = RunnableParallel(
    square=square,
    cube=cube,
    is_even=is_even
)

parallel_chain.invoke(3)

{'square': 9, 'cube': 27, 'is_even': False}

In [17]:
parallel_chain.batch([1, 2, 3, 4, 5])

[{'square': 1, 'cube': 1, 'is_even': False},
 {'square': 4, 'cube': 8, 'is_even': True},
 {'square': 9, 'cube': 27, 'is_even': False},
 {'square': 16, 'cube': 64, 'is_even': True},
 {'square': 25, 'cube': 125, 'is_even': False}]

## RunnablePassthrough
- `RunnablePassthrough`은 입력을 그대로 통과시키는 Runnable이다.
- 단독으로 사용하면 특별한 변환을 하지 않지만, 딕셔너리 조합과 함께 사용하면 원본 입력을 유지한채 다른 결과를 함께 만들어 낼 수 있다.

In [18]:
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()

passthrough.invoke("원본 입력")

'원본 입력'

In [19]:
# 원본 입력과 가공 결과를 함께 반환
chain = RunnableParallel(
    original=RunnablePassthrough(),
    length=RunnableLambda(lambda text: len(text)),
    upper=RunnableLambda(lambda text: text.upper())
)

chain.invoke("langchain")

{'original': 'langchain', 'length': 9, 'upper': 'LANGCHAIN'}

In [20]:
# assign()을 사용하면 기존 딕셔너리 입력에 새 값을 추가할 수 있다.
chain = RunnablePassthrough.assign(
    total=lambda data: data["price"] * data["quantity"]
)

chain.invoke({"name" : "노트북", "price" : 1200000, "quantity" : 2})

{'name': '노트북', 'price': 1200000, 'quantity': 2, 'total': 2400000}